# Part 2: Add grounding from web results

In Part 1, you built a knowledge base with indexed data. Now you'll connect a knowledge base to the new MAI grounding service which provides results from the web. That way, your knowledge base can provide answers for questions that require up-to-date news or topics outside the internal data.

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
MAI_GROUNDING_KEY = os.environ["MAI_GROUNDING_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

def azure_search_api_url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{AZURE_SEARCH_SERVICE_ENDPOINT}{path}{sep}api-version={AZURE_SEARCH_API_VERSION}"

session = requests.Session()
session.headers.update({"api-key": AZURE_SEARCH_ADMIN_KEY, "Accept": "application/json"})


print("Environment variables loaded")

## Step 2: Create three knowledge sources

For this knowledge base, you'll first create three knowledge sources - the same two index-based sources as you made in the first notebook, plus an additional knowledge source for the MAI grounding MCP server:

* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `web-knowledge-source`: Points to the remote MCP server for the MAI grounding service, using key-based authentication and `web` tool.

In [ ]:
HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowlege-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

def create_search_index_source(name: str, index_name: str, description: str):
    body = {
        "name": name,
        "description": description,
        "kind": "searchIndex",
        "searchIndexParameters": {
            "searchIndexName": index_name,
            "sourceDataFields": [{"name": "uid"}, {"name": "snippet_parent_id"}, {"name": "blob_path"}, {"name": "snippet"}],
            "searchFields": [{"name": "snippet"}],
            "semanticConfigurationName": "semantic-configuration",
        },
    }
    r = session.put(azure_search_api_url(f"/knowledgesources('{name}')"), json=body, headers={"Prefer": "return=representation"})
    r.raise_for_status()


create_search_index_source(HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "LAB532 HR documents")
create_search_index_source(HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "LAB532 health benefits documents")

web_knowledge_source_body = {"name": WEB_KNOWLEDGE_SOURCE_NAME,
                             "kind": "mcpServer",
                             "description": "LAB532 MAI Grounding MCP knowledge source",
                             "mcpServerParameters":
                             {"serverURL": "https://api.microsoft.ai/v3/mcp",
                              "authentication": {"kind": "storedHeaders",
                                                 "storedHeadersParameters":
                                                 {"headers": {"x-apikey": MAI_GROUNDING_KEY}}},
                              "tools": [{"name": "web", "outputParsing": {"kind": "auto"}}]}}
r = session.put(azure_search_api_url(f"/knowledgesources('{WEB_KNOWLEDGE_SOURCE_NAME}')"),
                json=web_knowledge_source_body, headers={"Prefer":"return=representation"})

r.raise_for_status()
print("Knowledge sources created")

## Step 3: Create the multi-source + web knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format response

For this notebook, we are using an `outputMode` of "answerSynthesis" so that the attached LLM can also answer the original user query. We are also using a `retrievalReasoningEffort` of "low", which means that the LLM will be used for query planning and knowledge source selection.

In [ ]:
KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

body = {"name": KNOWLEDGE_BASE_NAME,
        "description": "LAB532 KB combining restored indexes and MAI Grounding MCP",
        "outputMode": "answerSynthesis",
        "retrievalReasoningEffort": {"kind": "low"},
        "knowledgeSources": [{"name": n} for n in KNOWLEDGE_SOURCE_NAMES],
        "models": [{"kind": "azureOpenAI",
                    "azureOpenAIParameters": {
                        "resourceUri": AZURE_OPENAI_ENDPOINT + "/",
                        "apiKey": AZURE_OPENAI_KEY,
                        "deploymentId": AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                        "modelName": AZURE_OPENAI_CHATGPT_MODEL_NAME}}],
        "retrievalInstructions": "Use the restored HR and health indexes for company policy questions. Use MAI Grounding for current public web context."}
r = session.put(azure_search_api_url(f"/knowledgebases('{KNOWLEDGE_BASE_NAME}')"),
                json=body, headers={"Prefer": "return=representation"})
r.raise_for_status()
print("Knowledge base created")

## Step 4: Query the knowledge base

Since this knowledge base has the ability to search the web, ask a question that requires web results:

"Answer with citations: what employee health benefits are described in the company docs, and what is Azure AI Search knowledge base preview?"

The knowledge base uses agentic retrieval to:

1. Analyze the query to understand two different topics
2. Decompose the query into focused subqueries
3. Determine which knowledge sources are relevant for each subquery
4. Run searches concurrently against the selected sources
5. Merge the results with the semanting re-ranking model

In [ ]:
from IPython.display import Markdown, display

retrieve_body = {
    "includeActivity": True,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Answer with citations: what employee health benefits are described in the company docs, and what is Azure AI Search knowledge base preview?",
                }
            ],
        }
    ],
    "knowledgeSourceParams": [
        {
            "kind": "searchIndex",
            "knowledgeSourceName": HR_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        },
        {
            "kind": "searchIndex",
            "knowledgeSourceName": HEALTH_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        },
        {
            "kind": "mcpServer",
            "knowledgeSourceName": WEB_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
        },
    ],
    "maxRuntimeInSeconds": 120,
}
r = session.post(
    azure_search_api_url(f"/knowledgebases('{KNOWLEDGE_BASE_NAME}')/retrieve"),
    json=retrieve_body,
    timeout=180,
 )
r.raise_for_status()
result = r.json()
display(Markdown(result["response"][0]["content"][0]["text"]))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references shows which sources were retrieved, and the source metadata for each reference. Every result contains a `rerankerScore` from the re-ranking step. The results from the index include the search index fields, and the results from MAI grounding contain the title, URL, and truncated webpage snippet.

In [ ]:
print(json.dumps(result.get("references", []), indent=2))

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

➡️ Continue to: [Part 3: Add Fabric IQ](part3-fabric-iq-to-kb.ipynb).